# Silver weekly player statistics

## Pipeline overview

**Purpose:** Preserve nflverse weekly statistics as clean, source-shaped Silver records for all positions, regular-season weeks, and postseason weeks. This notebook does not calculate fantasy features or change existing downstream notebooks.

| Role | Files | Columns | Grain |
| --- | --- | ---: | --- |
| Data input | `data/bronze/player_stats_weekly/player_stats_weekly_{season}.parquet`, 2023–2026 | 150 per season | Source player-game-team record |
| Validation only | `data/silver/schedules/silver_schedules_{season}.parquet` | 46 per season | `game_id` |
| Output | `data/silver/player_stats_weekly/silver_player_stats_weekly_{season}.parquet` | 150 per season | `player_id + game_id + team`, including null IDs |

**Cleaning:** Remove exact full-row duplicates; fail on conflicting source keys; cast text fields to nullable strings; retain existing integer and float measures, all source columns, and source nulls. The `player_id`-null rows are unattributed team-level statistics, not identified players.

**Blocking checks:** Source schema and season; unique source grain after exact deduplication; non-null game/team/week context; valid game context against Silver schedules; finite numeric values; selected count and arithmetic checks; exact Bronze-value reconciliation; and Parquet round trip.

**Diagnostics, not repairs:** Null-player-ID rows, negative source yards or fantasy points, and shares outside 0–1. Air-yard shares can legitimately be negative or exceed one because the source denominator is signed. Row counts are observed, not hard-coded, because 2026 is in progress.


## 1. Set scope and contracts

The Bronze numeric fields are already integer or float. Text is normalized explicitly; source statistics and source-derived ratios remain unchanged.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

SEASONS = [2023, 2024, 2025, 2026]
EXPECTED_COLUMNS = 150
KEY_COLUMNS = ["player_id", "game_id", "team"]
CONTEXT_COLUMNS = ["season", "week", "season_type", "game_id", "team", "opponent_team"]
TEXT_COLUMNS = [
    "player_id", "player_name", "player_display_name", "position",
    "position_group", "headshot_url", "season_type", "game_id",
    "team", "opponent_team", "fg_made_list",
    "fg_missed_list", "fg_blocked_list",
]
COUNT_COLUMNS = [
    "completions", "attempts", "passing_tds", "passing_interceptions",
    "sacks_suffered", "carries", "rushing_tds",
    "receptions", "targets", "receiving_tds",
    "fg_made", "fg_att", "pat_made", "pat_att",
    "penalties", "def_safeties",
]

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "ROADMAP.md").exists():
    assert PROJECT_ROOT != PROJECT_ROOT.parent, "Open this notebook inside the project checkout"
    PROJECT_ROOT = PROJECT_ROOT.parent

BRONZE_DIR = PROJECT_ROOT / "data" / "bronze" / "player_stats_weekly"
SCHEDULE_DIR = PROJECT_ROOT / "data" / "silver" / "schedules"
OUTPUT_DIR = PROJECT_ROOT / "data" / "silver" / "player_stats_weekly"


## 2. Inspect source schemas and load each season

Names and column order must agree across partitions. The source row count may change on refresh, but the 150-column contract must remain explicit.


In [2]:
source_columns = None
bronze_partitions = {}
source_inventory = []

for season in SEASONS:
    source_path = BRONZE_DIR / f"player_stats_weekly_{season}.parquet"
    schedule_path = SCHEDULE_DIR / f"silver_schedules_{season}.parquet"
    assert source_path.exists(), f"Missing Bronze input: {source_path}"
    assert schedule_path.exists(), f"Missing schedule validation input: {schedule_path}"

    column_names = pq.read_schema(source_path).names
    assert len(column_names) == EXPECTED_COLUMNS, (season, len(column_names))
    assert set(TEXT_COLUMNS + CONTEXT_COLUMNS + COUNT_COLUMNS).issubset(column_names)
    if source_columns is None:
        source_columns = column_names
    else:
        assert column_names == source_columns, f"Schema drift in {source_path}"

    bronze = pd.read_parquet(source_path)
    assert bronze.columns.tolist() == source_columns
    assert bronze["season"].eq(season).all(), f"Unexpected season in {source_path}"
    bronze_partitions[season] = bronze
    source_inventory.append({
        "season": season, "bronze_rows": len(bronze),
        "columns": len(bronze.columns), "source": str(source_path.relative_to(PROJECT_ROOT)),
    })

pd.DataFrame(source_inventory)


,season,bronze_rows,columns,source
0,2023,18643,150,data/bronze/player_stats_weekly/player_stats_w...
1,2024,18983,150,data/bronze/player_stats_weekly/player_stats_w...
2,2025,19422,150,data/bronze/player_stats_weekly/player_stats_w...
3,2026,5506,150,data/bronze/player_stats_weekly/player_stats_w...


## 3. Deduplicate exact rows and normalize text

A null `player_id` is a known source state. It stays null; a different row at the same nullable key still stops the build rather than being silently chosen or aggregated.


In [3]:
deduplicated_partitions = {}
silver_partitions = {}
duplicate_counts = {}

for season in SEASONS:
    bronze = bronze_partitions[season]
    exact_duplicates = bronze.duplicated(keep="first")
    duplicate_counts[season] = int(exact_duplicates.sum())
    source = bronze.loc[~exact_duplicates].reset_index(drop=True)
    assert not source.duplicated(KEY_COLUMNS).any(), (
        season, source.loc[source.duplicated(KEY_COLUMNS, keep=False), KEY_COLUMNS].head(10)
    )

    silver = source.copy()
    for column in TEXT_COLUMNS:
        silver[column] = silver[column].astype("string")

    assert silver.columns.tolist() == source_columns
    assert len(silver) == len(bronze) - duplicate_counts[season]
    deduplicated_partitions[season] = source
    silver_partitions[season] = silver

pd.DataFrame([
    {"season": season, "exact_duplicates_removed": duplicate_counts[season],
     "silver_rows": len(silver_partitions[season])}
    for season in SEASONS
])


,season,exact_duplicates_removed,silver_rows
0,2023,0,18643
1,2024,0,18983
2,2025,0,19422
3,2026,0,5506


## 4. Validate types and source statistics

Negative yardage, negative fantasy points, and out-of-range air-yard shares can be valid. Only selected counts and basic within-row arithmetic have nonnegative or ordering rules.


In [4]:
for season in SEASONS:
    silver = silver_partitions[season]
    assert silver[CONTEXT_COLUMNS].notna().all().all(), season
    assert silver["season"].eq(season).all(), season
    assert silver["week"].between(1, 23).all(), season
    assert silver["season_type"].isin(["REG", "POST"]).all(), season
    assert not silver.duplicated(KEY_COLUMNS).any(), season

    for column in silver.select_dtypes(include="number").columns:
        assert np.isfinite(silver[column].dropna().to_numpy()).all(), (season, column)
    for column in COUNT_COLUMNS:
        assert pd.api.types.is_integer_dtype(silver[column]), (season, column, silver[column].dtype)
        assert silver[column].ge(0).all(), (season, column)

    for made, attempts in [
        ("completions", "attempts"), ("receptions", "targets"),
        ("fg_made", "fg_att"), ("pat_made", "pat_att"),
    ]:
        assert silver[made].le(silver[attempts]).all(), (season, made, attempts)

    source = deduplicated_partitions[season]
    for column in source_columns:
        expected = source[column].astype("string") if column in TEXT_COLUMNS else source[column]
        pd.testing.assert_series_equal(silver[column], expected, check_names=False)

print("Type, statistic, key, and Bronze-value checks passed for all seasons.")


Type, statistic, key, and Bronze-value checks passed for all seasons.


## 5. Reconcile game context with Silver schedules

Schedules are validation-only: they provide no output columns. `WC`, `DIV`, `CON`, and `SB` correspond to weekly-stat `POST`.


In [5]:
for season in SEASONS:
    schedule_path = SCHEDULE_DIR / f"silver_schedules_{season}.parquet"
    schedules = pd.read_parquet(
        schedule_path,
        columns=["game_id", "season", "week", "game_type", "home_team", "away_team"],
    )
    assert schedules["game_id"].notna().all()
    assert not schedules.duplicated("game_id").any()

    game_teams = silver_partitions[season][CONTEXT_COLUMNS].drop_duplicates()
    checked = game_teams.merge(
        schedules, on="game_id", how="left",
        suffixes=("", "_schedule"), validate="many_to_one", indicator=True,
    )
    assert checked["_merge"].eq("both").all(), season
    assert checked["season"].eq(checked["season_schedule"]).all(), season
    assert checked["week"].eq(checked["week_schedule"]).all(), season
    assert checked["season_type"].eq(
        checked["game_type"].where(checked["game_type"].eq("REG"), "POST")
    ).all(), season
    valid_opponents = (
        checked["team"].eq(checked["home_team"])
        & checked["opponent_team"].eq(checked["away_team"])
    ) | (
        checked["team"].eq(checked["away_team"])
        & checked["opponent_team"].eq(checked["home_team"])
    )
    assert valid_opponents.all(), season

print("Every weekly-stat game/team context matches Silver schedules.")


Every weekly-stat game/team context matches Silver schedules.


## 6. Review source exceptions

These diagnostics preserve source evidence. The null-ID rows are not treated as player participation, and none of the displayed values are changed.


In [6]:
diagnostics = []
for season in SEASONS:
    silver = silver_partitions[season]
    unidentified = silver["player_id"].isna()
    diagnostics.append({
        "season": season,
        "unidentified_rows": int(unidentified.sum()),
        "unidentified_penalties": int(silver.loc[unidentified, "penalties"].sum()),
        "unidentified_safeties": int(silver.loc[unidentified, "def_safeties"].sum()),
        "negative_receiving_yards": int(silver["receiving_yards"].lt(0).sum()),
        "negative_fantasy_points": int(silver["fantasy_points"].lt(0).sum()),
        "air_yards_share_outside_zero_one": int((
            silver["air_yards_share"].lt(0) | silver["air_yards_share"].gt(1)
        ).sum()),
    })

diagnostic_summary = pd.DataFrame(diagnostics)
diagnostic_summary


,season,unidentified_rows,unidentified_penalties,unidentified_safeties,negative_receiving_yards,negative_fantasy_points,air_yards_share_outside_zero_one
0,2023,22,264,10,80,132,801
1,2024,22,307,6,58,113,759
2,2025,22,294,3,73,107,732
3,2026,5,121,1,24,32,203


## 7. Persist and verify the four partitions

All in-memory partitions have passed validation before the first write. Reloaded Parquet files must reproduce the complete 150-column Silver frames.


In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
validation_rows = []

for season in SEASONS:
    silver = silver_partitions[season]
    output_path = OUTPUT_DIR / f"silver_player_stats_weekly_{season}.parquet"
    silver.to_parquet(output_path, index=False)
    reloaded = pd.read_parquet(output_path)
    pd.testing.assert_frame_equal(reloaded, silver)
    assert reloaded.columns.tolist() == source_columns
    assert len(reloaded.columns) == EXPECTED_COLUMNS
    assert not reloaded.duplicated(KEY_COLUMNS).any()
    validation_rows.append({
        "season": season,
        "bronze_rows": len(bronze_partitions[season]),
        "exact_duplicates_removed": duplicate_counts[season],
        "silver_rows": len(reloaded),
        "columns": len(reloaded.columns),
        "output": str(output_path.relative_to(PROJECT_ROOT)),
    })

validation_summary = pd.DataFrame(validation_rows)
assert validation_summary["silver_rows"].eq(
    validation_summary["bronze_rows"] - validation_summary["exact_duplicates_removed"]
).all()
validation_summary


,season,bronze_rows,exact_duplicates_removed,silver_rows,columns,output
0,2023,18643,0,18643,150,data/silver/player_stats_weekly/silver_player_...
1,2024,18983,0,18983,150,data/silver/player_stats_weekly/silver_player_...
2,2025,19422,0,19422,150,data/silver/player_stats_weekly/silver_player_...
3,2026,5506,0,5506,150,data/silver/player_stats_weekly/silver_player_...
